In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import os 
import glob
#import astroalign as aa
from astropy.io import fits
try:
    from photutils import aperture_photometry, CircularAperture
    from photutils.centroids import centroid_quadratic
    
except:
    pass
import processing as  pr


In [ ]:
data_dir = "/Users/admin/M83/"
output_dir = "/Users/admin/M83/"

## 1. Calibration du master bias

#### recuperer la liste des poses individuelles de bias

In [ ]:
list_bias_name = glob.glob('/Users/admin/M83/Bias/*')

#### generer le master bias et le sauver au format fits

In [ ]:
master_bias, master_bias_name = pr.master_bias(list_bias_name, out_dir =output_dir,
                             out_name = 'masterbias.fits', overwrite = 1)

#### afficher le master bias

In [ ]:
f = fits.open('/Users/admin/M83/masterbias.fits')
plt.imshow(f[0].data,vmin = np.mean(f[0].data)-np.std(f[0].data),vmax = np.mean(f[0].data)+np.std(f[0].data))

## 2. Ouvrir le master flat dans le filtre rouge

## 3. Traitement des images dans le filtre rouge

####  recuperer la liste des poses individuelles des images sciences

In [ ]:
list_images_red_name = glob.glob('/Users/admin/M83/*Red*fit')
print(len(list_images_red_name))

####  nom des fichiers contenant les poses maitres de calibration

In [ ]:
master_bias_name = '/Users/admin/M83/masterbias.fits'
master_flat_name_red = '/Users/admin/M83/master_flat--Red-T32.fits'

#### generer l'image finale et la sauvegarder au format fits

In [ ]:
final_image_red,final_image_red_name = pr.stacking(list_images_red_name, master_bias_name, master_flat_name_red
                           , out_dir = output_dir, out_name = 'image_finale_red.fits'
                           , overwrite = 1)

## 4. photométrie d'ouverture

In [ ]:
from photutils.detection import DAOStarFinder
from photutils import aperture_photometry, CircularAperture
from photutils.centroids import centroid_quadratic

positions = centroid_quadratic(final_image_red, xpeak=2040, ypeak=2200,search_boxsize = 400)
apertures = CircularAperture(positions, r=aperture_radius)  
